# ESMFold2 Local - Structure Prediction on Colab GPU

Run ESMFold2 **locally** on the Colab GPU to predict high-resolution 3D protein structures.

**Key Features:**
- Loads the ESMFold2 model directly onto the Colab GPU (no API, no Modal account)
- Predicts 3D structures from protein sequences
- Per-residue confidence (pLDDT) and overall pTM scores
- Interactive 3D visualization with py3Dmol
- Saves PDB structures + confidence metrics

**Compare with:**
- Notebook 4 (`ESMFold2_Biohub`): remote inference via Biohub API
- Notebook 5 (`ESMFold2_Modal`): serverless inference on Modal H100

**Requirements:** A Colab GPU runtime (Runtime -> Change runtime type -> GPU; A100 recommended for large proteins).

In [ ]:
!pip install -q torch transformers accelerate einops biotite biopython rdkit py3dmol matplotlib pandas numpy tqdm

In [ ]:
# Install ESM from GitHub
!pip install -q git+https://github.com/Biohub/esm.git@main

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import json
from tqdm import tqdm
from IPython.display import display, Markdown, HTML

# ESM imports
from esm.models import ESMFold2ForStructurePrediction
from esm.tokenization import TokenizerBase

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Enable GPU via Runtime -> Change runtime type -> GPU.")

## Configuration

In [ ]:
# Model & device
MODEL_NAME = "esmfold2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Output
OUTPUT_DIR = "/content/esmfold2_local_output"  # Change for local runs
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"Output directory: {OUTPUT_DIR}")

## Load ESMFold2 Model

In [ ]:
print(f"Loading {MODEL_NAME} onto {DEVICE}... (first run downloads weights, ~few min)")
model = ESMFold2ForStructurePrediction.from_pretrained("esm2/esmfold2").to(DEVICE)
model.eval()
tokenizer = TokenizerBase()
print(f"✓ Model loaded. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## Utility Functions

In [ ]:
def load_fasta(filepath: str) -> Dict[str, str]:
    """Load FASTA file into dictionary."""
    sequences = {}
    with open(filepath) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            sequences[current_id] = "".join(current_seq)
    return sequences

# 3-letter codes for PDB output
AA_3LETTER = {
    'A': 'ALA', 'R': 'ARG', 'N': 'ASN', 'D': 'ASP', 'C': 'CYS',
    'Q': 'GLN', 'E': 'GLU', 'G': 'GLY', 'H': 'HIS', 'I': 'ILE',
    'L': 'LEU', 'K': 'LYS', 'M': 'MET', 'F': 'PHE', 'P': 'PRO',
    'S': 'SER', 'T': 'THR', 'W': 'TRP', 'Y': 'TYR', 'V': 'VAL'
}

def coords_to_pdb(sequence: str, ca_coords: np.ndarray, plddt: np.ndarray,
                  chain_id: str = "A") -> str:
    """Write a CA-only backbone PDB string with pLDDT in the B-factor column."""
    lines = []
    for i, (aa, xyz) in enumerate(zip(sequence, ca_coords)):
        res = AA_3LETTER.get(aa, "UNK")
        b = float(min(100.0, max(0.0, plddt[i]))) if plddt is not None else 0.0
        x, y, z = float(xyz[0]), float(xyz[1]), float(xyz[2])
        lines.append(
            f"ATOM  {i+1:>5}  CA  {res} {chain_id}{i+1:>4}    "
            f"{x:>8.3f}{y:>8.3f}{z:>8.3f}  1.00{b:>6.2f}           C"
        )
    lines.append("TER")
    lines.append("END")
    return "\n".join(lines)

print("✓ Utility functions defined")

## Load Sequences

In [ ]:
#@markdown Upload a FASTA file or use the example sequences below
from google.colab import files

input_mode = "paste"  # "upload" or "paste"

if input_mode == "upload":
    print("Upload your FASTA file:")
    uploaded = files.upload()
    fasta_file = list(uploaded.keys())[0]
    sequences = load_fasta(fasta_file)
else:
    # Example proteins
    sequences = {
        "ubiquitin": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
        "gfp_short": "MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTG"
    }

print(f"Loaded {len(sequences)} sequences for structure prediction")
for seq_id, seq in sequences.items():
    print(f"  {seq_id}: {len(seq)} aa")

## Run Local Structure Prediction

In [ ]:
predictions = {}
errors = []

with torch.no_grad():
    for seq_id, sequence in tqdm(sequences.items(), desc="Predicting structures"):
        try:
            # Tokenize and run the model on the local GPU
            tokens = tokenizer.tokenize(sequence)
            tokens = torch.tensor([tokens]).to(DEVICE)
            output = model(tokens)

            # Extract CA coordinates and confidence metrics
            # (field names follow the ESM SDK; adjust if your version differs)
            ca_coords = output.positions[0, :, 1, :].cpu().numpy()  # [L, 3] CA atoms
            plddt = (output.plddt[0].cpu().numpy()
                     if hasattr(output, "plddt") else np.full(len(sequence), 50.0))
            ptm = float(output.ptm) if hasattr(output, "ptm") else 0.0

            pdb_text = coords_to_pdb(sequence, ca_coords, plddt)

            predictions[seq_id] = {
                "sequence": sequence,
                "length": len(sequence),
                "pdb": pdb_text,
                "plddt": plddt,
                "ptm": ptm,
            }
            print(f"  ✓ {seq_id}: mean pLDDT {np.mean(plddt):.1f}, pTM {ptm:.3f}")
        except Exception as e:
            print(f"  ✗ {seq_id} failed: {e}")
            errors.append((seq_id, str(e)))

print(f"\n✓ Completed {len(predictions)}/{len(sequences)} predictions")
if errors:
    print(f"✗ Failed: {', '.join(s for s, _ in errors)}")

## Confidence Metrics (pLDDT)

In [ ]:
import matplotlib.pyplot as plt

if predictions:
    fig, axes = plt.subplots(len(predictions), 1, figsize=(14, 4*len(predictions)))
    if len(predictions) == 1:
        axes = [axes]

    for idx, (seq_id, pred) in enumerate(predictions.items()):
        ax = axes[idx]
        plddt = pred["plddt"]
        ax.plot(plddt, linewidth=2, color="steelblue")
        ax.fill_between(range(len(plddt)), plddt, alpha=0.3, color="steelblue")
        ax.set_ylim([0, 100])
        ax.axhspan(90, 100, alpha=0.1, color="green")
        ax.axhspan(70, 90, alpha=0.1, color="yellow")
        ax.axhspan(50, 70, alpha=0.1, color="orange")
        ax.axhspan(0, 50, alpha=0.1, color="red")
        ax.set_title(f"{seq_id} - pLDDT (mean {np.mean(plddt):.1f})")
        ax.set_xlabel("Residue Position")
        ax.set_ylabel("pLDDT")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/confidence_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

    display(Markdown("**pLDDT:** 90-100 very high · 70-90 high · 50-70 medium · <50 low"))

## 3D Structure Visualization

In [ ]:
try:
    import py3Dmol

    for seq_id, pred in predictions.items():
        view = py3Dmol.view(width=800, height=500)
        view.addModel(pred["pdb"], "pdb")
        # Color by pLDDT-style spectrum along the backbone
        view.setStyle({}, {"cartoon": {"color": "spectrum"}})
        view.zoomTo()
        display(HTML(f"<h3>{seq_id} - Predicted Structure (CA backbone)</h3>"))
        display(view.show())
except ImportError:
    print("py3Dmol not available - structures are still saved as PDB files below.")

## Save Results

In [ ]:
for seq_id, pred in predictions.items():
    # PDB structure
    with open(f"{OUTPUT_DIR}/{seq_id}_structure.pdb", "w") as f:
        f.write(pred["pdb"])

    # Confidence metrics
    metrics = {
        "sequence_id": seq_id,
        "length": pred["length"],
        "mean_plddt": float(np.mean(pred["plddt"])),
        "ptm": pred["ptm"],
        "model": MODEL_NAME,
        "execution": "local-colab-gpu",
    }
    with open(f"{OUTPUT_DIR}/{seq_id}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

print(f"✓ All results saved to {OUTPUT_DIR}/")
print("  - Structures: {seq_id}_structure.pdb")
print("  - Metrics: {seq_id}_metrics.json")
print("  - Confidence plot: confidence_analysis.png")

## Summary

In [ ]:
summary_data = []
for seq_id, pred in predictions.items():
    plddt = pred["plddt"]
    summary_data.append({
        "Protein": seq_id,
        "Length (aa)": pred["length"],
        "Mean pLDDT": round(float(np.mean(plddt)), 1),
        "pTM": round(pred["ptm"], 3),
        "High-conf residues (pLDDT>70)": int(np.sum(plddt > 70)),
    })

df_summary = pd.DataFrame(summary_data)
display(df_summary)
df_summary.to_csv(f"{OUTPUT_DIR}/prediction_summary.csv", index=False)
print(f"\n✓ Summary saved to {OUTPUT_DIR}/prediction_summary.csv")